# PowerIt (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/PowerIt_python.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `PowerIt`: l'algoritmo in Python, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/PowerIt_cpp.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/PowerIt_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_python.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Python 3, senza installare nulla.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fib_rec` (snake_case).

## Potenza veloce: ricorsiva e PowerIt (square-and-multiply)

- **FastPower**$(x,n)$ (capitolo 14) — ricorsiva: $x^n=(x^{\lfloor n/2\rfloor})^2$ se $n$ è pari, $(x^{\lfloor n/2\rfloor})^2\cdot x$ se è dispari. $\Theta(\log_2 n)$ passi, $\lfloor\log_2 n\rfloor+2$ chiamate (profondità $\Theta(\log_2 n)$). Il risultato della chiamata ricorsiva è salvato in $y$ e usato due volte.
- **PowerIt**$(x,n)$ — *Input*: un numero $x$ e un naturale $n$. *Output*: $x^n$. Legge i bit di $n$ dal meno significativo: $\lfloor\log_2 n\rfloor+1$ iterazioni, memoria extra $\Theta(1)$; vale $p\cdot y^k=x^n$ a ogni iterazione.

Con $x=2$ e $n=13=(1101)_2$ si stampa la traccia del capitolo: $p$ viene moltiplicato per $x$, $x^4$ e $x^8$ (i bit di 13 uguali a 1) e $p=x^{13}$ dopo $4$ iterazioni. Poi la tabella: stesso risultato, $\lfloor\log_2 n\rfloor+1$ iterazioni contro $\lfloor\log_2 n\rfloor+2$ chiamate, contro le $n$ moltiplicazioni del ciclo naïf.

In [1]:
pow_calls = 0   # chiamate di FastPower
pow_iters = 0   # iterazioni di PowerIt

def fast_power(x, n):
    """FastPower(x, n): x^n per ricorsione sul dimezzamento di n."""
    global pow_calls
    pow_calls += 1
    if n == 0:
        return 1                    # caso base
    y = fast_power(x, n // 2)       # una sola chiamata ricorsiva
    if n % 2 == 0:
        return y * y                # n pari
    return x * y * y                # n dispari

def power_it(x, n):
    """PowerIt(x, n): x^n con square-and-multiply."""
    global pow_iters
    p = 1
    y = x
    k = n
    while k > 0:
        pow_iters += 1
        if k % 2 == 1:
            p = p * y
        y = y * y
        k = k // 2
    return p

def power_it_trace(x, n):
    print(f"PowerIt({x}, {n}): {n} = ({bin(n)[2:]})_2")
    p = 1
    y = x
    k = n
    it = 0
    while k > 0:
        it += 1
        par = "dispari" if k % 2 == 1 else "pari"
        if k % 2 == 1:
            p = p * y
        y = y * y
        print(f"{it}. k={k} {par}: p={p}, y={y}, k={k // 2}")
        k = k // 2
    print(f"iterazioni = {it}, risultato p = {p}")

power_it_trace(2, 13)
print()
print(f"{'n':>3s} {'2^n':>12s} {'PowerIt':>8s} {'FastPower':>10s} {'naive':>6s}   (iterazioni, chiamate, moltiplicazioni)")
for n in [0, 1, 2, 5, 13, 30]:
    pow_iters = 0
    r1 = power_it(2, n)
    pow_calls = 0
    r2 = fast_power(2, n)
    print(f"{n:>3d} {r1:>12d} {pow_iters:>8d} {pow_calls:>10d} {n:>6d}   {'OK' if r1 == r2 == 2 ** n else 'ERROR'}")

PowerIt(2, 13): 13 = (1101)_2
1. k=13 dispari: p=2, y=4, k=6
2. k=6 pari: p=2, y=16, k=3
3. k=3 dispari: p=32, y=256, k=1
4. k=1 dispari: p=8192, y=65536, k=0
iterazioni = 4, risultato p = 8192

  n          2^n  PowerIt  FastPower  naive   (iterazioni, chiamate, moltiplicazioni)
  0            1        0          1      0   OK
  1            2        1          2      1   OK
  2            4        2          3      2   OK
  5           32        3          4      5   OK
 13         8192        4          5     13   OK
 30   1073741824        5          6     30   OK


**Prova tu.** Prova un'altra base e un altro esponente (in C++ e Java il risultato deve stare in 64 bit).

In [2]:
# Prova un'altra base e un altro esponente (attenzione: in C++ e Java il risultato deve stare in 64 bit).
pow_iters = 0
pow_calls = 0
print(power_it(3, 25), pow_iters, fast_power(3, 25), pow_calls)

847288609443 5 847288609443 6


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).